# GPUs
:label:`sec_use_gpu`

In :numref:`tab_intro_decade`, we discussed the rapid growth
of computation over the past two decades.
In a nutshell, GPU performance has increased
by a factor of 1000 every decade since 2000.
This offers great opportunities but it also suggests
a significant need to provide such performance.


In this section, we begin to discuss how to harness
this computational performance for your research.
First by using single GPUs and at a later point,
how to use multiple GPUs and multiple servers (with multiple GPUs).

Specifically, we will discuss how
to use a single NVIDIA GPU for calculations.
First, make sure you have at least one NVIDIA GPU installed.
Then, download the [NVIDIA driver and CUDA](https://developer.nvidia.com/cuda-downloads)
and follow the prompts to set the appropriate path.
Once these preparations are complete,
the `nvidia-smi` command can be used
to (**view the graphics card information**).


In [1]:
!nvidia-smi

Fri Dec 12 02:09:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In PyTorch, every array has a device, we often refer it as a context.
So far, by default, all variables
and associated computation
have been assigned to the CPU.
Typically, other contexts might be various GPUs.
Things can get even hairier when
we deploy jobs across multiple servers.
By assigning arrays to contexts intelligently,
we can minimize the time spent
transferring data between devices.
For example, when training neural networks on a server with a GPU,
we typically prefer for the model's parameters to live on the GPU.


To run the programs in this section,
you need at least two GPUs.
Note that this might be extravagant for most desktop computers
but it is easily available in the cloud, e.g.,
by using the AWS EC2 multi-GPU instances.
Almost all other sections do *not* require multiple GPUs.
Instead, this is simply to illustrate
how data flow between different devices.


In [2]:
import torch
from torch import nn

## [**Computing Devices**]

We can specify devices, such as CPUs and GPUs,
for storage and calculation.
By default, tensors are created in the main memory
and then use the CPU to calculate it.


In PyTorch, the CPU and GPU can be indicated by `torch.device('cpu')` and `torch.device('cuda')`.
It should be noted that the `cpu` device
means all physical CPUs and memory.
This means that PyTorch's calculations
will try to use all CPU cores.
However, a `gpu` device only represents one card
and the corresponding memory.
If there are multiple GPUs, we use `torch.device(f'cuda:{i}')`
to represent the $i^\mathrm{th}$ GPU ($i$ starts from 0).
Also, `gpu:0` and `gpu` are equivalent.


In [3]:
def cpu():
    """Get the CPU device."""
    return torch.device('cpu')

def gpu(i=0):
    """Get a GPU device."""
    return torch.device(f'cuda:{i}')

cpu(), gpu(), gpu(1)

(device(type='cpu'),
 device(type='cuda', index=0),
 device(type='cuda', index=1))

We can (**query the number of available GPUs.**)


In [4]:
def num_gpus():
    """Get the number of available GPUs."""
    return torch.cuda.device_count()

num_gpus()

1

Now we [**define two convenient functions that allow us
to run code even if the requested GPUs do not exist.**]


In [5]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus():
    """Return all available GPUs, or [cpu(),] if no GPU exists."""
    return [gpu(i) for i in range(num_gpus())]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

## Tensors and GPUs


By default, tensors are created on the CPU.
We can [**query the device where the tensor is located.**]


In [6]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

It is important to note that whenever we want
to operate on multiple terms,
they need to be on the same device.
For instance, if we sum two tensors,
we need to make sure that both arguments
live on the same device---otherwise the framework
would not know where to store the result
or even how to decide where to perform the computation.

### Storage on the GPU

There are several ways to [**store a tensor on the GPU.**]
For example, we can specify a storage device when creating a tensor.
Next, we create the tensor variable `X` on the first `gpu`.
The tensor created on a GPU only consumes the memory of this GPU.
We can use the `nvidia-smi` command to view GPU memory usage.
In general, we need to make sure that we do not create data that exceeds the GPU memory limit.


In [7]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

Assuming that you have at least two GPUs, the following code will (**create a random tensor on the second GPU.**)


In [8]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y

tensor([[0.1647, 0.1375, 0.5179],
        [0.8447, 0.5270, 0.1642]])

### Copying

[**If we want to compute `X + Y`,
we need to decide where to perform this operation.**]
For instance, as shown in :numref:`fig_copyto`,
we can transfer `X` to the second GPU
and perform the operation there.
*Do not* simply add `X` and `Y`,
since this will result in an exception.
The runtime engine would not know what to do:
it cannot find data on the same device and it fails.
Since `Y` lives on the second GPU,
we need to move `X` there before we can add the two.

![Copy data to perform an operation on the same device.](../img/copyto.svg)
:label:`fig_copyto`


In [10]:
Z = X.cpu()
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')


Now that [**the data is on the same GPU
(both `Z` and `Y` are),
we can add them up.**]


In [12]:
X + Z

tensor([[2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')

Imagine that your variable `Z` already lives on your second GPU.
What happens if we still call `Z.cuda(1)`?
It will return `Z` instead of making a copy and allocating new memory.


In [13]:
Z.cuda(0) is Z

True

### Side Notes

People use GPUs to do machine learning
because they expect them to be fast.
But transferring variables between devices is slow.
So we want you to be 100% certain
that you want to do something slow before we let you do it.
If the deep learning framework just did the copy automatically
without crashing then you might not realize
that you had written some slow code.

Also, transferring data between devices (CPU, GPUs, and other machines)
is something that is much slower than computation.
It also makes parallelization a lot more difficult,
since we have to wait for data to be sent (or rather to be received)
before we can proceed with more operations.
This is why copy operations should be taken with great care.
As a rule of thumb, many small operations
are much worse than one big operation.
Moreover, several operations at a time
are much better than many single operations interspersed in the code
unless you know what you are doing.
This is the case since such operations can block if one device
has to wait for the other before it can do something else.
It is a bit like ordering your coffee in a queue
rather than pre-ordering it by phone
and finding out that it is ready when you are.

Last, when we print tensors or convert tensors to the NumPy format,
if the data is not in the main memory,
the framework will copy it to the main memory first,
resulting in additional transmission overhead.
Even worse, it is now subject to the dreaded global interpreter lock
that makes everything wait for Python to complete.


## [**Neural Networks and GPUs**]

Similarly, a neural network model can specify devices.
The following code puts the model parameters on the GPU.


In [14]:
net = nn.Sequential(nn.LazyLinear(1))
net = net.to(device=try_gpu())

We will see many more examples of
how to run models on GPUs in the following chapters,
simply since they will become somewhat more computationally intensive.

When the input is a tensor on the GPU, the model will calculate the result on the same GPU.


In [15]:
net(X)

tensor([[-0.4707],
        [-0.4707]], device='cuda:0', grad_fn=<AddmmBackward0>)

Let's (**confirm that the model parameters are stored on the same GPU.**)


In [16]:
net[0].weight.data.device

device(type='cuda', index=0)

Let the trainer support GPU.


In [18]:
import inspect
from typing import Iterable, Optional, Any, Tuple
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F


from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from IPython import display

import torchvision
from torchvision import transforms

def add_to_class(Class):
    """Decorator: add a function as a method to an existing class.

    Example:
        @add_to_class(MyClass)
        def new_method(self, x):
            ...
    """
    def wrapper(func):
        setattr(Class, func.__name__, func)
        return func
    return wrapper

class HyperParameters:
    """Base class that auto-saves __init__ arguments into `self.hparams`."""

    def save_hyperparameters(self, ignore=None):
        if ignore is None:
            ignore = []
        frame = inspect.currentframe().f_back
        args = frame.f_locals
        self.hparams = {}
        for k, v in args.items():
            if k != "self" and k not in ignore and not k.startswith("_"):
                setattr(self, k, v)
                self.hparams[k] = v

class Module(torch.nn.Module, HyperParameters):
    """Minimal D2L-like Module with a default loss & training_step."""

    def __init__(self):
        super().__init__()

    def loss(self, y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Default L2 loss assuming matching shapes."""
        return (y_hat - y) ** 2 / 2

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError("forward() not implemented")

    def __call__(self, X: torch.Tensor) -> torch.Tensor:
        return super().__call__(X)

    def training_step(self, batch) -> torch.Tensor:
        """Default training step: compute loss on a batch (X, y)."""
        X, y = batch
        l = self.loss(self(X), y)
        return l.mean()

    def validation_step(self, batch) -> torch.Tensor:
        """Optional validation step; by default, same as training loss."""
        X, y = batch
        l = self.loss(self(X), y)
        return l.mean()

    def plot(self, name, value, train=True):
        if isinstance(value, torch.Tensor):
            value = value.detach().cpu().item()

        if not hasattr(self, "history"):
            self.history = {"train": {}, "val": {}}

        key = "train" if train else "val"
        metric_dict = self.history[key].setdefault(name, [])
        metric_dict.append(value)


class SGD(HyperParameters):
    """Minibatch stochastic gradient descent for a list of tensors."""

    def __init__(self, params: Iterable[torch.Tensor], lr: float):
        self.save_hyperparameters()
        # `params` is stored by save_hyperparameters as self.params

    def step(self):
        for param in self.params:
            if param.grad is not None:
                param.data -= self.lr * param.grad

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()

class DataModule(HyperParameters):  # @save
    """The base class of data."""
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train: bool):
        """To be implemented by subclasses."""
        raise NotImplementedError

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)

    def get_tensorloader(self, tensors, train: bool, indices=None):
        """Utility: create a DataLoader from a list of tensors.

        tensors: list/tuple of tensors with same first dimension
        train:   whether to shuffle
        indices: slice or index array to select subset
        """
        if indices is None:
            indices = slice(None)

        sliced = [t[indices] for t in tensors]
        dataset = TensorDataset(*sliced)
        return DataLoader(dataset,
                          batch_size=self.batch_size,
                          shuffle=train)

class Trainer(HyperParameters):
    """Minimal trainer that relies on model.configure_optimizers().

    Expected model interface:
        - model(X): forward pass
        - model.training_step(batch): returns a scalar loss tensor
        - model.configure_optimizers(): returns an optimizer-like object
          with .zero_grad() and .step()
    """

    def __init__(self, max_epochs: int = 3, gradient_clip_val: float = 0.0):
        self.save_hyperparameters()

    def fit(self, model: Module, data: DataModule):
        self.model = model
        self.data = data
        self.model.train()

        # Ask the model for its optimizer (works for scratch + nn versions)
        self.optim = self.model.configure_optimizers()

        train_iter = data.train_dataloader()
        val_iter = data.val_dataloader() if hasattr(data, "val_dataloader") else None

        for epoch in range(self.max_epochs):
            # Training loop
            train_losses = []
            for batch in train_iter:
                loss = self.model.training_step(batch)
                self.optim.zero_grad()
                loss.backward()

                if self.gradient_clip_val and self.gradient_clip_val > 0:
                    self.clip_gradients(self.gradient_clip_val, self.model)

                self.optim.step()
                train_losses.append(loss.item())

            # Validation (optional)
            val_loss = None
            if val_iter is not None:
                self.model.eval()
                with torch.no_grad():
                    val_losses = []
                    for batch in val_iter:
                        l = self.model.validation_step(batch)
                        val_losses.append(l.item())
                val_loss = sum(val_losses) / len(val_losses)
                self.model.train()

            # Logging
            train_loss = sum(train_losses) / len(train_losses)
            if val_loss is not None:
                print(
                    f"epoch {epoch + 1}, train loss {train_loss:.6f}, "
                    f"val loss {val_loss:.6f}"
                )
            else:
                print(f"epoch {epoch + 1}, train loss {train_loss:.6f}")

    @staticmethod
    def clip_gradients(max_norm: float, model: Module):
        """Gradient clipping by global norm."""
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)


In [19]:
@add_to_class(Trainer)
def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
    self.save_hyperparameters()
    self.gpus = [d2l.gpu(i) for i in range(min(num_gpus, d2l.num_gpus()))]

@add_to_class(Trainer)
def prepare_batch(self, batch):
    if self.gpus:
        batch = [a.to(self.gpus[0]) for a in batch]
    return batch

@add_to_class(Trainer)
def prepare_model(self, model):
    model.trainer = self
    model.board.xlim = [0, self.max_epochs]
    if self.gpus:
        model.to(self.gpus[0])
    self.model = model

In short, as long as all data and parameters are on the same device, we can learn models efficiently. In the following chapters we will see several such examples.

## Summary

We can specify devices for storage and calculation, such as the CPU or GPU.
  By default, data is created in the main memory
  and then uses the CPU for calculations.
The deep learning framework requires all input data for calculation
  to be on the same device,
  be it CPU or the same GPU.
You can lose significant performance by moving data without care.
  A typical mistake is as follows: computing the loss
  for every minibatch on the GPU and reporting it back
  to the user on the command line (or logging it in a NumPy `ndarray`)
  will trigger a global interpreter lock which stalls all GPUs.
  It is much better to allocate memory
  for logging inside the GPU and only move larger logs.
